# Análisis LLM-as-a-Judge — Grupo 14 (IIC3633)

Consolida los 10 archivos `judge_output_path` (5 métodos × 2 datasets) generados por `llm_judge.ipynb` (uno por integrante) y produce:
1. Tabla cuantitativa de promedios por método/dataset/métrica (va directo al paper).
2. Análisis cualitativo: peores casos, contraste automático mejor-vs-peor método, taxonomía de patrones de fallo.

**Requisito:** completa `RUNS` abajo con los 10 `judge_output_path` reales (uno por método-dataset), y ajusta `JUDGE_K` al N final acordado por el equipo.

**No corre ninguna llamada a la API** — todo opera sobre los JSON ya guardados.

## 1. Setup

In [1]:
import os, json
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [11]:
from google.colab import  drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Config

In [12]:

JUDGE_K = 60

RUNS = [
    # pearl
    {"method": "cot",        "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/cot_pearl_judge.json"},
    {"method": "reflexion",  "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/reflexion_pearl_judge.json"},
    {"method": "dc",         "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/dc_pearl_judge.json"},
    {"method": "ace_base",   "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/ace_base_pearl_judge.json"},
    {"method": "ace_tools",  "dataset": "pearl", "judge_output_path": PROJECT_DIR + "judge/ace_tools_pearl_judge.json"},
    # redial
    {"method": "cot",        "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/cot_redial_judge.json"},
    {"method": "reflexion",  "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/reflexion_redial_judge.json"},
    {"method": "dc",         "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/dc_redial_judge.json"},
    {"method": "ace_base",   "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/ace_base_redial_judge.json"},
    {"method": "ace_tools",  "dataset": "redial", "judge_output_path": PROJECT_DIR + "judge/ace_tools_redial_judge.json"},
]

## 3. Carga con truncado a K

Se toman los primeros `k` registros guardados en cada archivo.

In [21]:
def load_scores(path, k=JUDGE_K):
    if not os.path.exists(path):
        print(f"⚠️ No existe: {path}")
        return pd.DataFrame()
    df = pd.DataFrame(json.load(open(path)))
    if df.empty:
        return df
    return df.head(k).reset_index(drop=True)

run_lookup = {(r["method"], r["dataset"]): r for r in RUNS}

## 4. Cuantitativo — tabla de promedios (va al paper)

In [24]:
def summarize_all_runs(runs, k=JUDGE_K, save_csv=PROJECT_DIR+"judge/summary_scores.csv"):
    rows = []
    for run_cfg in runs:
        method, dataset = run_cfg["method"], run_cfg["dataset"]
        df = load_scores(run_cfg["judge_output_path"], k=k)
        if df.empty:
            continue
        n = len(df)
        rows.append({
            "method": method,
            "dataset": dataset,
            "n": n,
            "relevance_mean": df["relevance_score"].mean(),
            "relevance_std": df["relevance_score"].std(),
            "coherence_mean": df["coherence_score"].mean(),
            "coherence_std": df["coherence_score"].std(),
            "explainability_mean": df["explainability_score"].mean(),
            "explainability_std": df["explainability_score"].std(),
        })
    summary = pd.DataFrame(rows).sort_values(["dataset", "method"]).reset_index(drop=True)
    os.makedirs(os.path.dirname(save_csv) or ".", exist_ok=True)
    summary.to_csv(save_csv, index=False)
    print(f"Guardado en {save_csv}")
    return summary

In [25]:
summary_df = summarize_all_runs(RUNS)
summary_df

Guardado en /content/drive/MyDrive/Proyecto RecSys/judge/summary_scores.csv


,method,dataset,n,relevance_mean,relevance_std,coherence_mean,coherence_std,explainability_mean,explainability_std
0,ace_base,pearl,60,4.483333,0.999859,4.583333,0.907439,4.733333,0.860955
1,ace_tools,pearl,57,4.421053,0.962648,4.561404,0.707550,4.771930,0.707550
2,cot,pearl,60,4.150000,1.400061,4.300000,1.279566,4.433333,1.240102
3,dc,pearl,58,4.275862,1.253670,4.413793,1.124441,4.603448,1.041922
4,reflexion,pearl,38,4.105263,1.351465,4.421053,0.976247,4.763158,0.714113
5,ace_base,redial,46,2.565217,1.796938,2.326087,1.415067,4.021739,1.164076
6,ace_tools,redial,49,3.367347,1.654462,2.530612,1.209374,3.714286,1.172604
7,cot,redial,60,3.516667,1.523729,4.216667,1.328841,3.916667,1.292635
8,dc,redial,58,3.793103,1.239109,2.465517,1.173023,3.551724,1.062477
9,reflexion,redial,60,4.050000,1.141290,2.583333,1.211410,3.550000,1.294382


In [26]:
# Formato tabla pivotada para el paper
pivot = summary_df.pivot(index="method", columns="dataset",
                          values=["relevance_mean", "coherence_mean", "explainability_mean"])
pivot

relevance_mean           coherence_mean            \
dataset            pearl    redial          pearl    redial   
method                                                        
ace_base        4.483333  2.565217       4.583333  2.326087   
ace_tools       4.421053  3.367347       4.561404  2.530612   
cot             4.150000  3.516667       4.300000  4.216667   
dc              4.275862  3.793103       4.413793  2.465517   
reflexion       4.105263  4.050000       4.421053  2.583333   

          explainability_mean            
dataset                 pearl    redial  
method                                   
ace_base             4.733333  4.021739  
ace_tools            4.771930  3.714286  
cot                  4.433333  3.916667  
dc                   4.603448  3.551724  
reflexion            4.763158  3.550000

## 5. Cualitativo — peores casos por run (incluye ACE, no solo el peor método)

In [19]:
def worst_cases(run_cfg, metric="relevance_score", n=3, k=JUDGE_K):
    df = load_scores(run_cfg["judge_output_path"], k=k)
    if df.empty:
        return df
    reasoning_col = metric.replace("_score", "_reasoning")
    return df.sort_values(metric)[["idx", metric, reasoning_col]].head(n)

In [20]:


for run_cfg in RUNS:
    print(f"\n=== {run_cfg['method']}-{run_cfg['dataset']} ===")
    display(worst_cases(run_cfg, metric="relevance_score", n=2))


=== cot-pearl ===


,idx,relevance_score,relevance_reasoning
3,2,1,"The user already agreed to watch 'Extremely Wicked, Shockingly Evil and Vile' in the previous turn, so recommending it again violates the ANTI-ECHO RULE."
0,26,4,"The movie matches the user's request for a thought-provoking portrayal of a catastrophic event, emotional impact, and superb acting, though it is societal collapse rather than specifically nuclear war."



=== reflexion-pearl ===


,relevance_reasoning,relevance_score,coherence_reasoning,coherence_score,explainability_reasoning,explainability_score,idx



=== dc-pearl ===


...

,idx,relevance_score,relevance_reasoning
0,14,5,"The user explicitly stated they like movies made by Adam Sandler, and The Waterboy is a quintessential Adam Sandler film."



=== ace_base-redial ===


,idx,relevance_score,relevance_reasoning
1,1,1,"The user explicitly asked for comedies in the last turn and stated they were ready for movie night, but the assistant recommended horror movies instead."
2,2,1,"The first recommended title, The Waterboy (1998), was explicitly mentioned by the user earlier in the conversation, violating the ANTI-ECHO RULE."



=== ace_tools-redial ===


,idx,relevance_score,relevance_reasoning
0,0,1,"The user wanted movies like Super Troopers and American Pie (raunchy/slapstick comedies), but the assistant recommended a serious action thriller."
1,2,1,"The first recommended title, The Waterboy (1998), was explicitly mentioned by the user earlier in the conversation, violating the ANTI-ECHO RULE."


## 6. Contraste automático mejor-vs-peor método por dataset

Elección basada en el `summary_df` (data-driven, no a ojo): por cada dataset, el método con mayor media vs. el de menor media en `relevance_mean`.

In [ ]:
def pick_contrast_pair(summary_df, dataset, metric="relevance_mean"):
    sub = summary_df[summary_df["dataset"] == dataset]
    best = sub.loc[sub[metric].idxmax()]
    worst = sub.loc[sub[metric].idxmin()]
    return best, worst

def biggest_deltas(run_a, run_b, metric="relevance_score", n=3, k=JUDGE_K):
    df_a = load_scores(run_a["judge_output_path"], k=k)[["idx", metric]].rename(columns={metric: "a"})
    df_b = load_scores(run_b["judge_output_path"], k=k)[["idx", metric]].rename(columns={metric: "b"})
    merged = df_a.merge(df_b, on="idx")
    merged["delta"] = merged["a"] - merged["b"]
    return merged.sort_values("delta", ascending=False).head(n)

for dataset in summary_df["dataset"].unique():
    best, worst = pick_contrast_pair(summary_df, dataset, metric="relevance_mean")
    print(f"\n=== {dataset}: {best['method']} (mejor, {best['relevance_mean']:.2f}) vs {worst['method']} (peor, {worst['relevance_mean']:.2f}) ===")
    run_best = run_lookup[(best["method"], dataset)]
    run_worst = run_lookup[(worst["method"], dataset)]
    display(biggest_deltas(run_best, run_worst, metric="relevance_score", n=3))

## 7. Taxonomía de patrones de fallo (agregado, todo el experimento)

In [ ]:
def tag_failure_patterns(run_cfg, metric="relevance_score", threshold=2, k=JUDGE_K):
    df = load_scores(run_cfg["judge_output_path"], k=k)
    if df.empty:
        return {"echo": 0, "ignora_preferencia": 0, "justificacion_generica": 0}, df
    reasoning_col = metric.replace("_score", "_reasoning")
    low = df[df[metric] <= threshold]
    patterns = {
        "echo": low[reasoning_col].str.contains("ECHO|earlier|already mentioned", case=False, na=False).sum(),
        "ignora_preferencia": low[reasoning_col].str.contains("ignores|but the assistant recommended", case=False, na=False).sum(),
        "justificacion_generica": low[reasoning_col].str.contains("generic|not tied|not explicitly", case=False, na=False).sum(),
    }
    return patterns, low

pattern_rows = []
for run_cfg in RUNS:
    patterns, _ = tag_failure_patterns(run_cfg, metric="relevance_score", threshold=2)
    pattern_rows.append({"method": run_cfg["method"], "dataset": run_cfg["dataset"], **patterns})

pd.DataFrame(pattern_rows)

## 8. Siguiente paso (fuera de este notebook)

Con `summary_df` (paso 4) y los ejemplos elegidos de los pasos 5-6 como evidencia textual, correr Friedman + Wilcoxon post-hoc sobre el diseño pareado (mismos idx por dataset a través de los 5 métodos).

In [3]:
def check_idx_overlap(runs, dataset):
    idx_sets = {}
    for r in runs:
        if r["dataset"] != dataset:
            continue
        df = pd.DataFrame(json.load(open(r["judge_output_path"])))
        idx_sets[r["method"]] = set(df["idx"])

    common = set.intersection(*idx_sets.values())
    print(f"=== {dataset} ===")
    for m, s in idx_sets.items():
        print(f"  {m}: {len(s)} idx")
    print(f"  Intersección común a los 5 métodos: {len(common)} idx")
    return common

In [13]:
check_idx_overlap(RUNS, "pearl")

=== pearl ===
  cot: 160 idx
  reflexion: 38 idx
  dc: 58 idx
  ace_base: 293 idx
  ace_tools: 57 idx
  Intersección común a los 5 métodos: 0 idx


set()

In [14]:
check_idx_overlap(RUNS, "redial")

=== redial ===
  cot: 299 idx
  reflexion: 67 idx
  dc: 58 idx
  ace_base: 46 idx
  ace_tools: 49 idx
  Intersección común a los 5 métodos: 0 idx


set()

In [15]:
from scipy.stats import friedmanchisquare, wilcoxon, kruskal, mannwhitneyu
from itertools import combinations

def run_stats_redial(runs, dataset="redial", metric="relevance_score", k=JUDGE_K):
    data = {}
    for r in runs:
        if r["dataset"] != dataset:
            continue
        df = load_scores(r["judge_output_path"], k=k)
        data[r["method"]] = df.set_index("idx")[metric]
    common = set.intersection(*[set(s.index) for s in data.values()])
    aligned = {m: s.loc[sorted(common)].values for m, s in data.items()}
    print(f"N pareado real: {len(common)}")
    stat, p = friedmanchisquare(*aligned.values())
    print(f"Friedman: stat={stat:.3f}, p={p:.4f}")
    for a, b in combinations(aligned.keys(), 2):
        s, p = wilcoxon(aligned[a], aligned[b])
        print(f"  Wilcoxon {a} vs {b}: p={p:.4f}")

def run_stats_unpaired(runs, dataset="pearl", metric="relevance_score", k=JUDGE_K):
    groups = {}
    for r in runs:
        if r["dataset"] != dataset:
            continue
        df = load_scores(r["judge_output_path"], k=k)
        groups[r["method"]] = df[metric].values
        print(f"  {r['method']}: n={len(df)}")
    stat, p = kruskal(*groups.values())
    print(f"Kruskal-Wallis: stat={stat:.3f}, p={p:.4f}")
    for a, b in combinations(groups.keys(), 2):
        s, p = mannwhitneyu(groups[a], groups[b])
        print(f"  Mann-Whitney {a} vs {b}: p={p:.4f}")

In [18]:
run_stats_unpaired(RUNS)

  cot: n=4
  reflexion: n=0
  dc: n=1
  ace_base: n=58
  ace_tools: n=57
Kruskal-Wallis: stat=nan, p=nan
  Mann-Whitney cot vs reflexion: p=nan
  Mann-Whitney cot vs dc: p=1.0000
  Mann-Whitney cot vs ace_base: p=0.3411
  Mann-Whitney cot vs ace_tools: p=0.4884
  Mann-Whitney reflexion vs dc: p=nan
  Mann-Whitney reflexion vs ace_base: p=nan
  Mann-Whitney reflexion vs ace_tools: p=nan
  Mann-Whitney dc vs ace_base: p=0.2322
  Mann-Whitney dc vs ace_tools: p=0.3321
  Mann-Whitney ace_base vs ace_tools: p=0.4651


/tmp/ipykernel_1256/3674880442.py:28: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p = kruskal(*groups.values())
/tmp/ipykernel_1256/3674880442.py:31: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  s, p = mannwhitneyu(groups[a], groups[b])
